# Local CP2K unfolding test

Use this notebook to launch the local reproduction scripts and inspect the resulting NPZ files.

In [ ]:
from pathlib import Path
import subprocess
ROOT = Path.cwd().parent
CASES = ROOT / 'cases'
RUN = False
sorted(p.name for p in CASES.iterdir() if p.is_dir())

In [ ]:
case = 'problematic_4808'
# For hBN/graphene-style tests, choose one primitive basis, e.g. '1 2'.
# Atom indices are 1-based and can also use ranges such as '1..2'.
primitive_basis_atoms = '1 2'
basis_tol = 5e-2
mpi_ranks = 8
with_pdos = True

cmd = [
    'python', str(ROOT / 'scripts' / 'run_case.py'), case,
    '--basis-cluster-tol', str(basis_tol),
]
if primitive_basis_atoms.strip():
    cmd += ['--primitive-basis-atoms', primitive_basis_atoms]
if with_pdos:
    cmd.append('--pdos')
if mpi_ranks > 1:
    cmd += ['--mpi-ranks', str(mpi_ranks)]
' '.join(cmd)


In [ ]:
# Run this when ready. It can be slow for the large overlap logs.
if RUN:
    subprocess.run(cmd, check=True)

In [ ]:
npz = CASES / case / 'unfolding_bands.local.npz'
if npz.exists():
    subprocess.run(['python', str(ROOT / 'scripts' / 'inspect_unfolding_npz.py'), str(npz)], check=True)
else:
    print(f'Missing {npz}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from useful_notebooks_cp2k_unfolding.plotting import plot_unfolded_kpath

npz = CASES / case / 'unfolding_bands.local.npz'
spin = 1
emin, emax = -18, 6
marker_scale = 220

with np.load(npz) as data:
    energies = data[f'evals_ev_spin_{spin}'] - float(data['ref_energy_ev'])
    weights = data[f'weights_spin_{spin}']
    mask = (energies >= emin) & (energies <= emax)
    disp_key = f'atom_mapping_displacements_spin_{spin}'
    if disp_key in data:
        disp = np.linalg.norm(data[disp_key], axis=1)
        print(f'atom mapping displacement max [A]: {disp.max():.6f}')
        print(f'atom mapping displacement mean [A]: {disp.mean():.6f}')
        print(f'atom mapping worst atom [1-based]: {int(np.argmax(disp)) + 1}')
    if 'primitive_basis_atom_indices' in data:
        print('primitive basis atom indices:', data['primitive_basis_atom_indices'])

    ax = plot_unfolded_kpath(
        path_k_indices=data['path_k_indices'],
        path_x=data['path_x'],
        x_ticks=data['x_ticks'],
        x_tick_labels=[str(x) for x in data['x_tick_labels']],
        energies_ev=energies[mask],
        weights=weights[:, mask],
        marker_scale=marker_scale,
    )
    ax.set_ylim(emin, emax)
    ax.set_title(f'{case}: unfolded weights, spin {spin}')
    plt.show()


In [ ]:
# Optional: compare two NPZ files numerically.
# Example:
# a = CASES / 'working_4179' / 'unfolding_bands.npz'
# b = CASES / 'working_4179' / 'unfolding_bands.local.npz'
# with np.load(a) as da, np.load(b) as db:
#     for spin in (0, 1):
#         key = f'weights_spin_{spin}'
#         if key in da and key in db:
#             diff = np.max(np.abs(da[key] - db[key]))
#             print(key, diff)
